In [1]:
# 1. Clone repo và chuyển vào thư mục làm việc
!git clone https://github.com/theflyingkhui04/qwen2.5-7b-eagle3.1-speculator.git
%cd qwen2.5-7b-eagle3.1-speculator

# 2. Cài đặt các thư viện cần thiết
!pip install -r requirements.txt
!mkdir -p data/raw data/processed


Cloning into 'qwen2.5-7b-eagle3.1-speculator'...
remote: Enumerating objects: 91, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 91 (delta 31), reused 85 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (91/91), 43.15 KiB | 920.00 KiB/s, done.
Resolving deltas: 100% (31/31), done.
/content/qwen2.5-7b-eagle3.1-speculator


In [2]:
# Tải tập ShareGPT_Vicuna_unfiltered chuẩn từ Hugging Face
!wget -O data/raw/sharegpt.json https://huggingface.co/datasets/anon8231489123/ShareGPT_Vicuna_unfiltered/resolve/main/ShareGPT_V3_unfiltered_cleaned_split.json


--2026-10-11 04:19:37--  https://huggingface.co/datasets/anon8231489123/ShareGPT_Vicuna_unfiltered/resolve/main/ShareGPT_V3_unfiltered_cleaned_split.json
Resolving huggingface.co (huggingface.co)... 18.238.176.129, 18.238.176.103, 18.238.176.8, ...
Connecting to huggingface.co (huggingface.co)|18.238.176.129|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/642912f7a760fe0bf37996b1/d7094af68bb58022696728841937d7285db423eba9d055b3386797b28db2cbdb?user_id=public&response-content-type=application%2Fjson%3B+charset%3Dutf-8&X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27ShareGPT_V3_unfiltered_cleaned_split.json%3B+filename%3D%22ShareGPT_V3_unfiltered_cleaned_split.json%22%3B&xip=wvjBJbcCHlU&Expires=1791695977&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjQyOTEyZjdhNzYwZmUwYmYzNzk5NmIxL2Q3MDk0YWY2OGJiNTgwMjI2OTY3Mjg4NDE5MzdkNzI4NWRiNDIzZWJ

In [4]:
# 3. Tải trực tiếp 10.000 mẫu tiếng Việt từ bkai-foundation-models/vi-alpaca
from datasets import load_dataset
import json

print("Đang tải tập chỉ dẫn tiếng Việt bkai-foundation-models/vi-alpaca từ Hugging Face...")
ds = load_dataset("bkai-foundation-models/vi-alpaca", split="train")

formatted_samples = []
for item in ds:
    instruction = str(item.get("instruction", "")).strip()
    input_text = str(item.get("input", "")).strip()
    output_text = str(item.get("output", "")).strip()

    # Ghép instruction và input thành câu hỏi người dùng
    user_prompt = f"{instruction}\n{input_text}".strip() if input_text else instruction

    if user_prompt and output_text:
        formatted_samples.append({
            "conversations": [
                {"role": "user", "content": user_prompt},
                {"role": "assistant", "content": output_text}
            ]
        })

    # Lấy đủ 10.000 mẫu như yêu cầu đặc tả
    if len(formatted_samples) >= 10000:
        break

# Lưu vào thư mục data/raw/
with open("data/raw/vietnamese_invoices.json", "w", encoding="utf-8") as f:
    json.dump(formatted_samples, f, ensure_ascii=False, indent=2)

print(f"✅ Đã tải và lưu thành công {len(formatted_samples)} mẫu tiếng Việt vào data/raw/vietnamese_invoices.json")

Đang tải tập chỉ dẫn tiếng Việt bkai-foundation-models/vi-alpaca từ Hugging Face...


README.md:   0%|          | 0.00/1.74k [00:00<?, ?B/s]

data/train-00000-of-00001-b0855b79e84114(…): reconstructing file:   0%|          |  0.00B / 26.3MB            

data/train-00000-of-00001-b0855b79e84114(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/50006 [00:00<?, ? examples/s]

✅ Đã tải và lưu thành công 10000 mẫu tiếng Việt vào data/raw/vietnamese_invoices.json


In [6]:
# 4. Gộp và chuẩn hóa dữ liệu lai
!python data/prepare_hybrid_dataset.py \
    --sharegpt-path "data/raw/sharegpt.json" \
    --vi-path "data/raw/vietnamese_invoices.json" \
    --output-path "data/processed/hybrid_dataset.json" \
    --sharegpt-limit 20000 \
    --vi-limit 10000 \
    --seed 42

2026-10-11 04:26:42,834 - INFO - Đang nạp tập dữ liệu ShareGPT từ: data/raw/sharegpt.json
2026-10-11 04:26:58,783 - INFO - Đang nạp tập dữ liệu Tiếng Việt / MC-OCR từ: data/raw/vietnamese_invoices.json
2026-10-11 04:26:59,414 - INFO - Lọc được: 20000 mẫu ShareGPT, 10000 mẫu Tiếng Việt / MC-OCR
2026-10-11 04:27:02,719 - INFO - Đã ghi thành công 30000 mẫu vào data/processed/hybrid_dataset.json


In [7]:
# 5. Kiểm tra và tải file về máy tính cá nhân
import os, json
from google.colab import files

file_path = "data/processed/hybrid_dataset.json"

if os.path.exists(file_path):
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"✅ Tổng số mẫu trong tập lai: {len(data):,} mẫu")
    print(f"✅ Dung lượng file: {size_mb:.2f} MB")
    print("\n--- [Xem thử 1 mẫu dữ liệu] ---")
    print(json.dumps(data[0], ensure_ascii=False, indent=2)[:400] + "\n...")

    print("\n>>> Đang kích hoạt tải file về máy...")
    files.download(file_path)
else:
    print("❌ Chưa tìm thấy file data/processed/hybrid_dataset.json!")


✅ Tổng số mẫu trong tập lai: 30,000 mẫu
✅ Dung lượng file: 148.21 MB

--- [Xem thử 1 mẫu dữ liệu] ---
{
  "conversations": [
    {
      "role": "user",
      "content": "Write a crazy story and before you start, ask me for 6 words that you will use as an inspiration for the story. Make story as crazy and unusual as possible, but still believeable. give dialogue too. give detailed and imaginative descriptions of the charakters when they enter the scene"
    },
    {
      "role": "assistant",
    
...

>>> Đang kích hoạt tải file về máy...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>